### IMPORTING LIBRARIES

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller,kpss

### LOAD DATASET

In [34]:
df = pd.read_csv("../DATA/DATA[R].csv",  encoding='utf-8')

### CLEAN AND PREPROCESS DATA

In [8]:
df_new = pd.read_csv('../DATA/DATA[R].csv')

# IDENTIFY COLUMNS CONTAINING '--'
columns_with_dash = (df_new == '--').any(axis=0)  
columns_with_dash = columns_with_dash[columns_with_dash].index.tolist()

# COUNT OCCURRENCES OF '--'
dash_counts = (df_new == '--').sum(axis=0)

# REPLACE '--' WITH 0
df_clean = df_new.replace('--', 0)

# DROP NON-NUMERIC COLUMNS
df_conti = df_clean.drop(columns=['date.time', 'Prevailing Wind Direction', 'High Wind Direction'])

# DISPLAY COLUMN DETAILS
column_details = pd.DataFrame({
    'Column Name': df_conti.columns,
    'Data Type': df_conti.dtypes.values,
    'Null Values': df_conti.isnull().sum().values,
    'Non-Null Values': df_conti.notnull().sum().values
})

print(column_details)

          Column Name Data Type  Null Values  Non-Null Values
0        Inside Temp    float64            0            47839
1   High Inside Temp    float64            0            47839
2    Low Inside Temp    float64            0            47839
3         Inside Hum    float64            0            47839
4    High Inside Hum    float64            0            47839
..                ...       ...          ...              ...
64   High Wet Bulb .1   float64            0            47839
65    Low Wet Bulb .1   float64            0            47839
66      Heat Index .1   float64            0            47839
67    High Heat Index   float64            0            47839
68        Unnamed: 71   float64            0            47839

[69 rows x 4 columns]


### CREATE DATAFRAMES FOR CORRELATION PAIRS

In [9]:
# COMPUTE PEARSON CORRELATION MATRIX
corr = df_conti.corr(method='pearson')

# FILTER CORRELATIONS (|CORRELATION| >= 0.5, EXCLUDING PERFECT CORRELATION)
filtered_corr = corr.where((corr.abs() >= 0.5) & (corr != 1.00))

# APPLY UPPER TRIANGULAR MASK TO REMOVE DUPLICATES
mask = np.triu(np.ones_like(filtered_corr, dtype=bool))
filtered_corr = filtered_corr.mask(mask)

# RECOMPUTE SIGNIFICANT CORRELATION PAIRS
pairs = []
for i in range(len(filtered_corr.index)):
    for j in range(i):
        corr_value = filtered_corr.iloc[i, j]
        if not pd.isna(corr_value):
            col1 = filtered_corr.index[i]
            col2 = filtered_corr.columns[j]
            pairs.append((col1, col2, corr_value))

# CREATE DATAFRAME FROM CORRELATION PAIRS
df_pairs = pd.DataFrame(pairs, columns=['Feature 1', 'Feature 2', 'Correlation'])

# FILTER PAIRS BASED ON REQUIRED CONDITIONS
filtered_pairs = [pair for pair in pairs if abs(pair[2]) >= 0.5]  # ADJUST CONDITION IF NECESSARY
df_filtered_pairs = pd.DataFrame(filtered_pairs, columns=['Feature 1', 'Feature 2', 'Correlation'])

### FILTER FEATURES BASED ON CORRELATION

In [10]:
# COMPUTE ABSOLUTE CORRELATION MATRIX
corr_matrix = df_conti.corr().abs()

# CONVERT MATRIX INTO LONG FORMAT FOR PAIR-WISE COMPARISON
df_pairs = corr_matrix.unstack().reset_index()
df_pairs.columns = ['FEATURE 1', 'FEATURE 2', 'CORRELATION']

# REMOVE SELF-CORRELATIONS
df_pairs = df_pairs[df_pairs["FEATURE 1"] != df_pairs["FEATURE 2"]]

# CREATE LOOKUP FOR CORRELATIONS
corr_lookup = {
    (row['FEATURE 2'], row['FEATURE 1']): row['CORRELATION']
    for _, row in df_pairs.iterrows()
}

# FUNCTION TO RETRIEVE CORRELATION VALUE
def get_correlation(f1, f2):
    return corr_lookup.get((f1, f2), 0)

# REMOVE HIGHLY CORRELATED FEATURES GREATER THAN 0.9
df_pairs["FEATURE XY"] = df_pairs.apply(
    lambda row: get_correlation(row["FEATURE 1"], row["FEATURE 2"]), axis=1
)
df_filtered_pairs = df_pairs[df_pairs["FEATURE XY"] < 0.9]

# FINAL SELECTED FEATURES
df_chosen = df_filtered_pairs.drop_duplicates(subset=['FEATURE 1', 'FEATURE 2'])

# DISPLAY SELECTED FEATURES
print(df_chosen)

         FEATURE 1              FEATURE 2  CORRELATION  FEATURE XY
3     Inside Temp             Inside Hum      0.256157    0.256157
4     Inside Temp        High Inside Hum      0.231651    0.231651
5     Inside Temp          Low Inside Hum     0.286396    0.286396
6     Inside Temp   Inside Dew Point - °C     0.793441    0.793441
8     Inside Temp              Barometer      0.057018    0.057018
...            ...                    ...          ...         ...
4754   Unnamed: 71       Low Dew Point .1     0.649695    0.649695
4755   Unnamed: 71            Wet Bulb .1     0.577705    0.577705
4756   Unnamed: 71       High Wet Bulb .1     0.750942    0.750942
4757   Unnamed: 71        Low Wet Bulb .1     0.776145    0.776145
4758   Unnamed: 71          Heat Index .1     0.712104    0.712104

[4366 rows x 4 columns]


### PREPARE RESIDUAL DATAFRAME FOR MODELING

In [35]:
# CREATE DATAFRAME WITH SELECTED FEATURES
residual_df = pd.DataFrame(
    data=df[['date.time', 'Barometer ', 'High AQI', 'PM 1', 'PM 2.5 ', 'PM 10 ', 'High Dew Point ', 'Temp .1', 'AQI']],
    columns=['date.time', 'Barometer ', 'High AQI', 'PM 1', 'PM 2.5 ', 'PM 10 ', 'High Dew Point ', 'Temp .1', 'AQI']
)

# STRIP WHITESPACE FROM COLUMN NAMES
residual_df.columns = residual_df.columns.str.strip()

# CREATE A COPY FOR BACKUP
residual_df_copy = residual_df.copy()

# SHIFT AQI COLUMN TO CREATE TARGET VARIABLE
residual_df['AQI_Target'] = residual_df['AQI'].shift(-3)

# REMOVE MISSING VALUES AFTER SHIFTING
residual_df = residual_df.dropna().reset_index(drop=True)

### IMPORTING MACHINE LEARNING LIBRARIES

In [21]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import xgboost as xgb
import lightgbm as lgb
import catboost as cb
import re

### FEATURE ENGINEERING: EXTRACTING TIME-BASED FEATURES

In [13]:
if 'date.time' in residual_df.columns:
    residual_df['date.time'] = pd.to_datetime(residual_df['date.time'], format='%d-%m-%Y %H:%M', errors='coerce')
    residual_df['hour'] = residual_df['date.time'].dt.hour
    residual_df['day'] = residual_df['date.time'].dt.day
    residual_df['month'] = residual_df['date.time'].dt.month
    residual_df.drop(columns=['date.time'], inplace=True)

### FEATURE ENGINEERING: EXTRACTING TIME-BASED FEATURES

In [14]:
if 'date.time' in df.columns:
    
    df['date.time'] = pd.to_datetime(df['date.time'], errors='coerce', format='mixed')
    
   
    df.dropna(subset=['date.time'], inplace=True)

    df['hour'] = df['date.time'].dt.hour
    df['day'] = df['date.time'].dt.day
    df['month'] = df['date.time'].dt.month

    df.drop(columns=['date.time'], inplace=True)

print("Date conversion successful!")

Date conversion successful!


### FEATURE ENGINEERING APPLIED TO RESIDUAL DATAFRAMES

- **TARGET VARIABLE ("AQI") EXTRACTED FROM THE RESIDUAL DATAFRAME**  
- **FEATURES PREPARED BY DROPPING TARGET COLUMN**  
- **CATEGORICAL VARIABLES ONE-HOT ENCODED USING `pd.get_dummies()`**  
- **ENSURES CONSISTENCY IN FEATURE PROCESSING ACROSS THE DATASET**  

In [15]:
Target = "AQI"  
Features = df .drop(columns=[Target])

Features = pd.get_dummies(Features)

### SPLITTING DATA INTO TRAINING & TESTING SETS

In [16]:
X_Train, X_Test, y_Train, y_Test = train_test_split(Features, df [Target], test_size=0.2, random_state=19)

# MODEL TRAINING AND EVALUATION

### RANDOM FOREST REGRESSOR

In [25]:
# FEATURE SCALING 
scaler = MinMaxScaler()
Features_scaled = scaler.fit_transform(Features)

# TRAIN-TEST SPLIT
X_Train, X_Test, y_Train, y_Test = train_test_split(
    Features_scaled, df[Target], test_size=0.2, random_state=19
)

# HYPERPARAMETER TUNING USING GRID SEARCH
param_grid = {
    "n_estimators": [100, 150],
    "max_depth": [10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt"]
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=19),
    param_grid,
    scoring="r2",
    cv=3,
    n_jobs=4  
)
grid_search.fit(X_Train, y_Train)

# TRAINING RANDOM FOREST REGRESSORS
best_params = grid_search.best_params_
Rf_model = RandomForestRegressor(**best_params, random_state=19)
Rf_model.fit(X_Train, y_Train)

# MAKING PREDICTIONS WITH RANDOM FOREST MODELS
Train_preds = Rf_model.predict(X_Train)
Test_preds = Rf_model.predict(X_Test)

# EVALUATE PERFORMANCE
Metrics = {
    "Train MSE": mean_squared_error(y_Train, Train_preds),
    "Train RMSE": np.sqrt(mean_squared_error(y_Train, Train_preds)),
    "Train MAE": mean_absolute_error(y_Train, Train_preds),
    "Train R²": r2_score(y_Train, Train_preds),
    "Test MSE": mean_squared_error(y_Test, Test_preds),
    "Test RMSE": np.sqrt(mean_squared_error(y_Test, Test_preds)),
    "Test MAE": mean_absolute_error(y_Test, Test_preds),
    "Test R²": r2_score(y_Test, Test_preds),
}

# PRINT METRICS
Metrics_df = pd.DataFrame(Metrics, index=[0])
print(Metrics_df)

    Train MSE  Train RMSE  Train MAE  Train R²    Test MSE  Test RMSE  \
0  159.705491   12.637464   8.160288  0.927753  208.934971  14.454583   

   Test MAE   Test R²  
0  8.687615  0.906679  


### XGBOOST REGRESSOR

In [30]:
# TRAIN XGBOOST MODEL
xgb_model = xgb.XGBRegressor(objective="reg:squarederror", n_estimators=500, learning_rate=0.05, max_depth=6, random_state=19)
xgb_model.fit(X_Train, y_Train)

# MAKE PREDICTIONS
Train_preds = xgb_model.predict(X_Train)
Test_preds = xgb_model.predict(X_Test)

# EVALUATE PERFORMANCE
Metrics = {
    "Train MSE": mean_squared_error(y_Train, Train_preds),
    "Train RMSE": np.sqrt(mean_squared_error(y_Train, Train_preds)),
    "Train MAE": mean_absolute_error(y_Train, Train_preds),
    "Train R²": r2_score(y_Train, Train_preds),
    "Test MSE": mean_squared_error(y_Test, Test_preds),
    "Test RMSE": np.sqrt(mean_squared_error(y_Test, Test_preds)),
    "Test MAE": mean_absolute_error(y_Test, Test_preds),
    "Test R²": r2_score(y_Test, Test_preds),
}

# PRINT METRICS
Metrics_df = pd.DataFrame(Metrics, index=[0])
print(Metrics_df)

   Train MSE  Train RMSE  Train MAE  Train R²  Test MSE  Test RMSE  Test MAE  \
0    0.11748    0.342754   0.185551  0.999947  0.326587   0.571478  0.235182   

    Test R²  
0  0.999854  


### LIGHTGBM REGRESSOR

In [31]:
# TRAIN LIGHTGBM MODEL
lgb_model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=19)
lgb_model.fit(X_Train, y_Train)

# MAKE PREDICTIONS
Train_preds = lgb_model.predict(X_Train)
Test_preds = lgb_model.predict(X_Test)

# EVALUATE PERFORMANCE
Metrics = {
    "Train MSE": mean_squared_error(y_Train, Train_preds),
    "Train RMSE": np.sqrt(mean_squared_error(y_Train, Train_preds)),
    "Train MAE": mean_absolute_error(y_Train, Train_preds),
    "Train R²": r2_score(y_Train, Train_preds),
    "Test MSE": mean_squared_error(y_Test, Test_preds),
    "Test RMSE": np.sqrt(mean_squared_error(y_Test, Test_preds)),
    "Test MAE": mean_absolute_error(y_Test, Test_preds),
    "Test R²": r2_score(y_Test, Test_preds),
}

# PRINT METRICS
Metrics_df = pd.DataFrame(Metrics, index=[0])
print(Metrics_df)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.616864 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21528
[LightGBM] [Info] Number of data points in the train set: 38271, number of used features: 7355
[LightGBM] [Info] Start training from score 66.313697
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive g

c:\WEATHER\MODEL\VENV\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\WEATHER\MODEL\VENV\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


   Train MSE  Train RMSE  Train MAE  Train R²  Test MSE  Test RMSE  Test MAE  \
0   0.779771    0.883046   0.226815  0.999647  4.905173   2.214762  0.270106   

    Test R²  
0  0.997809  


### CATBOOST REGRESSOR

In [32]:
# TRAIN CATBOOST MODEL
cb_model = cb.CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, random_state=19, verbose=0)
cb_model.fit(X_Train, y_Train)

# MAKE PREDICTIONS
Train_preds = cb_model.predict(X_Train)
Test_preds = cb_model.predict(X_Test)

# EVALUATE PERFORMANCE
Metrics = {
    "Train MSE": mean_squared_error(y_Train, Train_preds),
    "Train RMSE": np.sqrt(mean_squared_error(y_Train, Train_preds)),
    "Train MAE": mean_absolute_error(y_Train, Train_preds),
    "Train R²": r2_score(y_Train, Train_preds),
    "Test MSE": mean_squared_error(y_Test, Test_preds),
    "Test RMSE": np.sqrt(mean_squared_error(y_Test, Test_preds)),
    "Test MAE": mean_absolute_error(y_Test, Test_preds),
    "Test R²": r2_score(y_Test, Test_preds),
}

# PRINT METRICS
Metrics_df = pd.DataFrame(Metrics, index=[0])
print(Metrics_df)

   Train MSE  Train RMSE  Train MAE  Train R²  Test MSE  Test RMSE  Test MAE  \
0   0.867267    0.931272    0.66998  0.999608  1.064817   1.031899  0.694965   

    Test R²  
0  0.999524  


### COMPARISON WITHOUT SCALING

In [29]:
# FUNCTION TO CLEAN FEATURE NAMES
def clean_feature_names(df):
    df.columns = [re.sub(r"[^a-zA-Z0-9_]", "", col) for col in df.columns]
    return df

# SPLIT DATA
X_train, X_test, y_train, y_test = train_test_split(Features, df[Target], test_size=0.2, random_state=19)

# CLEAN FEATURE NAMES FOR LIGHTGBM
X_train = clean_feature_names(pd.DataFrame(X_train, columns=Features.columns))
X_test = clean_feature_names(pd.DataFrame(X_test, columns=Features.columns))

# CONVERT TO NUMPY ARRAYS FOR XGBOOST, LIGHTGBM, CATBOOST
X_train = X_train.values
X_test = X_test.values
y_train = y_train.values
y_test = y_test.values

# INITIALIZE MODELS
xgb_model = xgb.XGBRegressor(objective="reg:squarederror", n_estimators=500, learning_rate=0.05, max_depth=6, random_state=19)
lgb_model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=19)
cb_model = cb.CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, random_state=19, verbose=0)

# TRAIN MODELS
xgb_model.fit(X_train, y_train)
lgb_model.fit(X_train, y_train)
cb_model.fit(X_train, y_train)

# PREDICTIONS & METRICS
models = {"XGBoost": xgb_model, "LightGBM": lgb_model, "CatBoost": cb_model}
for name, model in models.items():
    y_pred = model.predict(X_test)
    
    print(f"\nModel: {name}")
    print("Test MSE:", mean_squared_error(y_test, y_pred))
    print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
    print("Test MAE:", mean_absolute_error(y_test, y_pred))
    print("Test R²:", r2_score(y_test, y_pred))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.510489 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21545
[LightGBM] [Info] Number of data points in the train set: 38271, number of used features: 7355
[LightGBM] [Info] Start training from score 66.313697
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive g

c:\WEATHER\MODEL\VENV\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



Model: LightGBM
Test MSE: 4.9051727490622525
Test RMSE: 2.2147624588344126
Test MAE: 0.2701057900120529
Test R²: 0.9978091112062426

Model: CatBoost
Test MSE: 1.0648165532297136
Test RMSE: 1.0318994879491479
Test MAE: 0.6949651408230804
Test R²: 0.9995244011224019
